# Task 2: held-out ligand case studies

Three cases per dataset selected for sufficient measured test candidates. Checkpoints are loaded, never refitted.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Run from the project root or notebooks directory'
sys.path.insert(0, str(ROOT))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from src.splits import load_splits
from src.cf import MatrixFactorization
from src.metrics import evaluate
from src.io import read_predictions
T = ROOT / 'results/tables'


In [2]:
for dataset in ['davis','kiba']:
    _, parts, manifest = load_splits(dataset, 'task2')
    model = MatrixFactorization.load(ROOT / f'saved_models/{dataset}_cf.npz')
    cases = pd.read_csv(T / f'task2_{dataset}_case_summary.csv')
    rankings = pd.read_csv(T / f'task2_{dataset}_case_rankings.csv')
    display(Markdown(f'## {dataset.upper()}'))
    display(cases[['original_drug_ids','selection_reason','test_candidates','true_best_predicted_rank','true_best_bias_rank','mse','ci_global','hit1','hit5','mrr']])
    for drug in cases.drug_id:
        query = parts['test'][parts['test'].drug_id == drug]
        ranked = rankings[rankings.drug_id == drug]
        lookup = dict(zip(query.target_id, model.predict(query)))
        np.testing.assert_allclose(ranked.prediction, ranked.target_id.map(lookup), atol=1e-12)
        assert ranked.target_id.isin(query.target_id).all()
        display(ranked.head(10))

## DAVIS

,original_drug_ids,selection_reason,test_candidates,true_best_predicted_rank,true_best_bias_rank,mse,ci_global,hit1,hit5,mrr
0,44259,Suggested source ID; >=10 measured test candid...,57,6,21,0.888857,0.798706,0.0,0.0,0.166667
1,11338033,Suggested source ID; >=10 measured test candid...,57,4,46,0.160516,0.969697,0.0,1.0,0.250000
2,10127622,Suggested source ID; >=10 measured test candid...,57,19,14,0.049747,0.678571,0.0,0.0,0.052632


,drug_id,target_id,original_drug_ids,original_target_ids,affinity,prediction,bias_prediction,predicted_rank,bias_rank,is_true_best,top1_correct
0,56,161,44259,JAK3(JH1domain-catalytic),7.920819,9.375492,7.546323,1,5,False,False
1,56,40,44259,CAMK2D,9.494850,8.765963,7.137085,2,27,False,False
2,56,180,44259,MAP3K2,8.619789,8.555557,7.365367,3,10,False,False
3,56,279,44259,PKN2,8.602060,8.468139,7.047788,4,37,False,False
4,56,27,44259,BMPR1B,6.769551,8.224869,7.352640,5,11,False,False
5,56,44,44259,CAMKK2,9.795880,8.215419,7.222896,6,21,True,False
6,56,173,44259,LRRK2|LRRK2(G2019S),8.958787,8.172938,7.245560,7,15,False,False
7,56,25,44259,BLK,7.823909,8.115660,7.571745,8,4,False,False
8,56,61,44259,CHEK1,8.494850,8.053645,7.047729,9,38,False,False
9,56,156,44259,IRAK3,7.886057,7.864840,7.235367,10,18,False,False


,drug_id,target_id,original_drug_ids,original_target_ids,affinity,prediction,bias_prediction,predicted_rank,bias_rank,is_true_best,top1_correct
57,3,53,11338033,CDK5,6.154902,6.384732,5.071332,1,47,False,False
58,3,292,11338033,PRKD3,5.000000,6.201140,5.234480,2,26,False,False
59,3,354,11338033,TNNI3K,5.000000,5.978421,5.304614,3,16,False,False
60,3,50,11338033,CDK2,7.522879,5.761454,5.078570,4,46,True,False
61,3,89,11338033,DMPK,5.958607,5.722572,5.238081,5,25,False,False
62,3,145,11338033,HIPK4,5.000000,5.652244,5.538581,6,6,False,False
63,3,295,11338033,PRKR,5.000000,5.589876,5.196322,7,33,False,False
64,3,285,11338033,PRKCD,5.000000,5.551697,5.246314,8,22,False,False
65,3,57,11338033,CDKL1,5.000000,5.508334,5.005590,9,53,False,False
66,3,150,11338033,IKK-alpha,5.000000,5.478938,5.205328,10,31,False,False


,drug_id,target_id,original_drug_ids,original_target_ids,affinity,prediction,bias_prediction,predicted_rank,bias_rank,is_true_best,top1_correct
114,54,68,10127622,CSF1R,5.0,5.885048,5.925176,1,2,False,False
115,54,253,10127622,PDGFRB,5.0,5.757273,6.267672,2,1,False,False
116,54,198,10127622,MEK5,5.0,5.541694,5.731592,3,6,False,False
117,54,252,10127622,PDGFRA,5.0,5.466118,5.740134,4,5,False,False
118,54,186,10127622,MAP4K5,5.0,5.340754,5.407398,5,11,False,False
119,54,300,10127622,RAF1,5.0,5.330242,5.066280,6,28,False,False
120,54,168,10127622,LCK,5.0,5.313751,5.724711,7,7,False,False
121,54,172,10127622,LOK,5.0,5.292577,5.632342,8,8,False,False
122,54,164,10127622,JNK3,5.0,5.266371,5.224896,9,20,False,False
123,54,54,10127622,CDK7,5.0,5.216822,5.256199,10,17,False,False


## KIBA

,original_drug_ids,selection_reason,test_candidates,true_best_predicted_rank,true_best_bias_rank,mse,ci_global,hit1,hit5,mrr
0,CHEMBL2004544,Suggested source ID; >=10 measured test candid...,15,2,3,0.099123,0.772277,0.0,1.0,0.5
1,CHEMBL565612,Suggested source ID; >=10 measured test candid...,25,1,22,0.485628,0.672414,1.0,1.0,1.0
2,CHEMBL2007574,Sparse/median/dense training history among eli...,10,1,9,0.041039,0.813953,1.0,1.0,1.0


,drug_id,target_id,original_drug_ids,original_target_ids,affinity,prediction,bias_prediction,predicted_rank,bias_rank,is_true_best,top1_correct
0,542,191,CHEMBL2004544,Q96PF2,12.199999,12.171101,11.767407,1,1,False,False
1,542,143,CHEMBL2004544,Q06418,12.499997,11.980180,11.718591,2,3,True,False
2,542,90,CHEMBL2004544,P42679,12.100000,11.969165,11.692805,3,4,False,False
3,542,11,CHEMBL2004544,O15530,11.500000,11.656903,11.547195,4,6,False,False
4,542,21,CHEMBL2004544,O95819,11.800001,11.574106,11.765896,5,2,False,False
5,542,31,CHEMBL2004544,P06213,11.400000,11.567937,11.439537,6,9,False,False
6,542,212,CHEMBL2004544,Q9NYL2,11.300000,11.535890,11.242908,7,14,False,False
7,542,206,CHEMBL2004544,Q9HBH9,11.100000,11.524596,11.440901,8,8,False,False
8,542,22,CHEMBL2004544,O96013,11.400000,11.419993,11.400640,9,11,False,False
9,542,199,CHEMBL2004544,Q9BZL6,11.699999,11.325004,11.417308,10,10,False,False


,drug_id,target_id,original_drug_ids,original_target_ids,affinity,prediction,bias_prediction,predicted_rank,bias_rank,is_true_best,top1_correct
15,644,132,CHEMBL565612,Q02156,14.307608,13.751583,11.655017,1,22,True,True
16,644,105,CHEMBL565612,P49840,12.899998,12.941330,11.990398,2,15,False,False
17,644,151,CHEMBL565612,Q13188,13.000009,12.859713,11.824280,3,18,False,False
18,644,182,CHEMBL565612,Q86V86,12.599997,12.779804,11.756361,4,20,False,False
19,644,121,CHEMBL565612,P53779,11.100000,12.604102,12.114473,5,4,False,False
20,644,2,CHEMBL565612,O00329,11.100000,12.531989,12.805892,6,1,False,False
21,644,146,CHEMBL565612,Q12851,11.400000,12.312255,12.020302,7,11,False,False
22,644,155,CHEMBL565612,Q13555,13.200015,12.220646,11.690692,8,21,False,False
23,644,23,CHEMBL565612,O96017,11.900001,12.156162,12.040479,9,9,False,False
24,644,183,CHEMBL565612,Q8IU85,11.900001,12.103783,11.845087,10,17,False,False


,drug_id,target_id,original_drug_ids,original_target_ids,affinity,prediction,bias_prediction,predicted_rank,bias_rank,is_true_best,top1_correct
40,576,56,CHEMBL2007574,P17612,13.800057,13.783230,12.252802,1,9,True,True
41,576,105,CHEMBL2007574,P49840,13.399989,13.585640,12.413638,2,6,False,False
42,576,136,CHEMBL2007574,Q04759,13.000009,13.224115,12.300134,3,7,False,False
43,576,161,CHEMBL2007574,Q14164,11.600000,12.013044,12.281703,4,8,False,False
44,576,22,CHEMBL2007574,O96013,11.699999,11.978093,12.130732,5,10,False,False
45,576,7,CHEMBL2007574,O14965,11.900001,11.897700,12.686352,6,3,False,False
46,576,40,CHEMBL2007574,P08069,11.699999,11.825574,12.452973,7,4,False,False
47,576,138,CHEMBL2007574,Q04912,11.800001,11.737690,12.433956,8,5,False,False
48,576,57,CHEMBL2007574,P17948,11.500000,11.734674,13.124205,9,1,False,False
49,576,39,CHEMBL2007574,P07949,11.600000,11.652383,12.873964,10,2,False,False


Target IDs can have exact-sequence aliases. The CSV lists actual source IDs and processed keys. All true-best ties are relevant; expected Hit/MRR average prediction ties. No training pairs appear in these tables.